In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '07_02_shared_activation_and_cross_task_roi_patterns.ipynb'
NOTEBOOK_RELATIVE_PATH = '07_02_shared_activation_and_cross_task_roi_patterns.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
from pathlib import Path as u12_Path
import json as u12_json
import os as u12_os
from pathlib import Path as u12_Path
import numpy as u12_np
import pandas as u12_pd
from pathlib import Path as u12_Path
u12_glm_TR = 2.0
u12_glm_FD_THRESHOLD = 0.5
u12_glm_STDDVARS_THRESHOLD = 1.5
u12_glm_STROOP_DURATION_S = 1.0
u12_glm_MOTION_BASE = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']


In [ ]:
def u12_glm_paths_for(subject, task, run=None):
    return u12_imaging_paths(subject, task, run)


In [ ]:
def u12_glm_load_events(path: u12_Path, task: str) -> u12_pd.DataFrame:
    ev = u12_pd.read_csv(path, sep=',' if u12_Path(path).suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    required = {'onset', 'duration', 'trial_type'}
    if not required.issubset(ev.columns):
        raise ValueError(f'Missing event columns {required - set(ev.columns)} in {path}')
    ev = ev[['onset', 'duration', 'trial_type'] + (['accuracy'] if 'accuracy' in ev.columns else [])].copy()
    if task == 'stroop':
        if 'accuracy' not in ev.columns:
            raise ValueError(f'Stroop accuracy column absent: {path}')
        accuracy = u12_pd.to_numeric(ev['accuracy'], errors='raise')
        if not accuracy.isin([0, 1]).all():
            raise ValueError('Stroop event accuracy must be observed 0/1')
        correct = accuracy.eq(1)
        ev.loc[~correct, 'trial_type'] = 'error'
    elif 'accuracy' in ev.columns and set(ev['trial_type'].astype(str).str.lower()).issubset({'go', 'nogo'}):
        accuracy = u12_pd.to_numeric(ev['accuracy'], errors='raise')
        if not accuracy.isin([0, 1]).all():
            raise ValueError('Go/No-Go event accuracy must be observed 0/1')
        acc = accuracy.eq(1)
        base = ev['trial_type'].astype(str).str.lower()
        ev['trial_type'] = u12_np.where(acc, base + '_correct', base + '_error')
    if 'accuracy' in ev.columns:
        ev = ev.drop(columns='accuracy')
    ev['trial_type'] = ev['trial_type'].astype(str).str.lower()
    ev['onset'] = u12_pd.to_numeric(ev['onset'], errors='raise')
    ev['duration'] = u12_pd.to_numeric(ev['duration'], errors='raise')
    allowed = {'congruent', 'neutral', 'incongruent', 'error'} if task == 'stroop' else {'go_correct', 'nogo_correct', 'go_error', 'nogo_error'}
    if not ev.trial_type.isin(allowed).all() or not u12_np.isfinite(ev[['onset', 'duration']].to_numpy(float)).all() or (ev.duration < 0).any():
        raise ValueError('Unknown event type or invalid event timing')
    if task == 'stroop':
        ev['duration'] = u12_glm_STROOP_DURATION_S
    if task == 'gonogo':
        ev['duration'] = 0.5
    return ev


In [ ]:
def u12_glm_raw_log_path(subject, task, run=None):
    return u12_glm_paths_for(subject, task, run)['events']


In [ ]:
def u12_glm_reconstruct_events(subject, task, run=None):
    return u12_glm_load_events(u12_glm_raw_log_path(subject, task, run), task)


In [ ]:
def u12_glm_load_confounds(path: u12_Path) -> tuple[u12_pd.DataFrame, dict[str, float | int]]:
    cf = u12_pd.read_csv(path, sep=',' if u12_Path(path).suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    required = {'framewise_displacement', 'std_dvars'}
    if not required.issubset(cf.columns):
        raise ValueError(f'Required spike-screening inputs absent: {required - set(cf.columns)}')
    selected: dict[str, u12_pd.Series] = {}
    for base in u12_glm_MOTION_BASE:
        for col in (base, f'{base}_derivative1', f'{base}_power2', f'{base}_derivative1_power2'):
            if col not in cf.columns:
                raise ValueError(f'Required motion confound absent: {col} in {path}')
            selected[col] = u12_pd.to_numeric(cf[col], errors='coerce').fillna(0.0)
    for col in cf.columns:
        if col.startswith('non_steady_state_outlier'):
            selected[col] = u12_pd.to_numeric(cf[col], errors='coerce').fillna(0.0)
    fd = u12_pd.to_numeric(cf.get('framewise_displacement', u12_pd.Series(0.0, index=cf.index)), errors='coerce').fillna(0.0)
    dv = u12_pd.to_numeric(cf.get('std_dvars', u12_pd.Series(0.0, index=cf.index)), errors='coerce').fillna(0.0)
    if float((fd > u12_glm_FD_THRESHOLD).mean()) > 0.2:
        raise ValueError('Run fails manuscript motion inclusion: more than 20% volumes with FD > 0.5 mm; revise the participant cohort before fitting')
    censor = (fd > u12_glm_FD_THRESHOLD) | (dv > u12_glm_STDDVARS_THRESHOLD)
    for idx in u12_np.flatnonzero(censor.to_numpy()):
        spike = u12_np.zeros(len(cf), dtype=float)
        spike[idx] = 1.0
        selected[f'motion_outlier_{idx:04d}'] = u12_pd.Series(spike)
    out = u12_pd.DataFrame(selected, index=cf.index).astype(float)
    info = {'n_volumes': int(len(cf)), 'n_confounds': int(out.shape[1]), 'n_spikes': int(censor.sum()), 'spike_pct': float(censor.mean() * 100.0), 'mean_fd_mm': float(fd.mean()), 'max_fd_mm': float(fd.max()), 'mean_std_dvars': float(dv.mean())}
    return (out, info)


In [ ]:
def u12_glm_acquisition(path: u12_Path) -> tuple[float, float]:
    meta = u12_json.loads(path.read_text(encoding='utf-8'))
    tr = float(meta.get('RepetitionTime', u12_glm_TR))
    start = float(meta.get('StartTime', 0.0))
    if not u12_np.isclose(tr, u12_glm_TR):
        raise ValueError(f'Unexpected TR={tr}: {path}')
    return (tr, start / tr)


In [ ]:
import json as u12_json
import os as u12_os
from pathlib import Path as u12_Path
import matplotlib as u12_matplotlib
import matplotlib.pyplot as u12_plt
from matplotlib import font_manager as u12_font_manager
import numpy as u12_np
import pandas as u12_pd
import json as u12_json
import numpy as u12_np
import pandas as u12_pd
import matplotlib.pyplot as u12_plt
import numpy as u12_np
import pandas as u12_pd
import json as u12_json
import numpy as u12_np
import matplotlib as u12_matplotlib
import matplotlib.pyplot as u12_plt
import numpy as u12_np
import os as u12_os
import matplotlib as u12_matplotlib
u12_matplotlib.use('Agg')
import matplotlib.pyplot as u12_plt
import numpy as u12_np
import pandas as u12_pd
u12_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
from pathlib import Path as u13_Path
import sys as u13_sys
from IPython.display import display as u13_display, Image as u13_Image
import sys as u13_sys
import argparse as u13_argparse
import json as u13_json
import nibabel as u13_nib
import numpy as u13_np
import pandas as u13_pd
from scipy import ndimage as u13_ndimage, stats as u13_stats
from nilearn.glm.first_level import make_first_level_design_matrix as u13_make_first_level_design_matrix


In [ ]:
def u13_alignment_overlap():
    root = u13_NEW
    zs = [u13_nib.load(root / f'task-{t}_analysis-primary_stat-z.nii.gz') for t in u13_CONTRAST]
    sig = []
    for (t, z) in zip(u13_CONTRAST, zs):
        lp = u13_nib.load(root / f'task-{t}_analysis-primary_stat-logpFWEclusterSize_p001.nii.gz')
        if z.shape != lp.shape or not u13_np.allclose(z.affine, lp.affine):
            raise ValueError('Group image geometries do not match')
        sig.append((u13_np.asarray(z.dataobj) > 0) & (u13_np.asarray(lp.dataobj) >= -u13_np.log10(0.05)))
    both = sig[0] & sig[1]
    out = u13_np.where(both, u13_np.minimum(u13_np.asarray(zs[0].dataobj), u13_np.asarray(zs[1].dataobj)), 0)
    u13_nib.save(u13_nib.Nifti1Image(out.astype('float32'), zs[0].affine), root / 'intersection_of_positive_cluster_FWE_maps.nii.gz')
    (labels, n) = u13_ndimage.label(both, u13_np.ones((3, 3, 3)))
    rows = []
    for i in range(1, n + 1):
        ijk = u13_np.argwhere(labels == i)
        peak = ijk[u13_np.argmax(out[tuple(ijk.T)])]
        xyz = u13_nib.affines.apply_affine(zs[0].affine, peak)
        rows.append(dict(cluster=i, voxels=len(ijk), volume_mm3=len(ijk) * abs(u13_np.linalg.det(zs[0].affine[:3, :3])), x=xyz[0], y=xyz[1], z=xyz[2], min_z=out[tuple(peak)]))
    u13_save_table(u13_pd.DataFrame(rows), u13_TASK, 'shared_positive_activation_corrected_map_intersection')
    u13_atomic_json(root / 'intersection_inference_notes.json', dict(voxels=int(both.sum()), components=n, operation='Intersection of separately corrected positive cluster-FWE maps', interpretation='Each voxel in the intersection belongs to a significant cluster in each task; the intersection components were not separately FWE-tested and do not constitute a voxelwise minimum-statistic conjunction'))
    print('Shared positive activation voxels', int(both.sum()), flush=True)


In [ ]:
def u13_alignment_dice():
    (masks, _) = u13_roi_masks()
    rows = []
    for sid in u13_subjects():
        (a, va, im, dfa) = u13_t_map(sid, 'stroop')
        (b, vb, _, dfb) = u13_t_map(sid, 'gonogo')
        valid = va & vb
        regions = {'whole_brain': valid}
        regions.update({r: masks[r] & valid for r in u13_ROIS[:3]})
        for p in (0.001, 0.01):
            (ca, cb) = (u13_stats.t.isf(p / 2, dfa), u13_stats.t.isf(p / 2, dfb))
            for direction in ('two_sided',):
                x = abs(a) > ca if direction == 'two_sided' else a > ca
                y = abs(b) > cb if direction == 'two_sided' else b > cb
                for (region, mask) in regions.items():
                    if region != 'whole_brain' and mask.sum() / masks[region].sum() < 0.8:
                        eligible = False
                    else:
                        eligible = True
                    na = int((x & mask).sum())
                    nb = int((y & mask).sum())
                    inter = int((x & y & mask).sum())
                    value = 2 * inter / (na + nb) if na + nb and eligible else u13_np.nan
                    rows.append(dict(subject_id=sid, region=region, p_uncorrected=p, direction=direction, eligible_80pct=eligible, stroop_voxels=na, gonogo_voxels=nb, intersection=inter, dice=value, overlap=inter > 0, neither_active=na + nb == 0, df_stroop=dfa, df_gonogo=dfb))
    df = u13_pd.DataFrame(rows)
    name = 'individual_Dice_whole_brain_and_shared_ROIs'
    u13_save_table(df, u13_TASK, name + '_subject_distribution')
    summary = []
    for (key, g) in df.groupby(['region', 'p_uncorrected', 'direction']):
        x = g.loc[g.eligible_80pct, 'dice'].dropna()
        eligible = g.eligible_80pct.sum()
        summary.append(dict(region=key[0], p_uncorrected=key[1], direction=key[2], N_eligible=int(eligible), N_dice_defined=len(x), N_neither_active=int(g.neither_active.sum()), mean=x.mean(), median=x.median(), Q25=x.quantile(0.25), Q75=x.quantile(0.75), N_overlap=int((g.overlap & g.eligible_80pct).sum()), proportion_overlap=float(g.loc[g.eligible_80pct, 'overlap'].mean())))
    u13_save_table(u13_pd.DataFrame(summary), u13_TASK, name + '_summary')
    print(name, 'complete', len(df), flush=True)


In [ ]:
import sys as u13_sys
import argparse as u13_argparse
import json as u13_json
import subprocess as u13_subprocess
import tempfile as u13_tempfile
import nibabel as u13_nib
import numpy as u13_np
import pandas as u13_pd
from scipy import stats as u13_stats
from scipy.optimize import minimize as u13_minimize, brentq as u13_brentq
from nilearn.glm.first_level import FirstLevelModel as u13_FirstLevelModel, run_glm as u13_run_glm, make_first_level_design_matrix as u13_make_first_level_design_matrix
from nilearn.glm.first_level.first_level import mean_scaling as u13_mean_scaling
from nilearn.glm.contrasts import compute_fixed_effects as u13_compute_fixed_effects, compute_contrast as u13_compute_contrast
from nilearn.masking import apply_mask as u13_apply_mask


In [ ]:
def u13_roi_prepare_roi(source, reference, destination, transforms=()):
    (source, reference, destination) = map(u13_Path, (source, reference, destination))
    im = u13_nib.load(source)
    a = u13_np.asarray(im.dataobj)
    if not u13_np.isin(a, [0, 1]).all() or not a.any():
        raise ValueError('Source ROI must be nonempty and binary')
    ref = u13_nib.load(reference)
    transform_paths = [u13_Path(p) for p in transforms]
    if any((not p.is_file() for p in transform_paths)):
        raise FileNotFoundError('An ANTs transform file is missing')
    destination.parent.mkdir(parents=True, exist_ok=True)
    with u13_tempfile.TemporaryDirectory(prefix='a8_roi_') as temp:
        if transform_paths:
            output = u13_Path(temp) / 'mapped.nii.gz'
            command = ['antsApplyTransforms', '-d', '3', '-i', str(source), '-r', str(reference), '-o', str(output), '-n', 'GenericLabel']
            for transform in transform_paths:
                command.extend(['-t', str(transform)])
            u13_subprocess.run(command, check=True, capture_output=True, text=True)
            im = u13_nib.load(output)
        a = u13_np.asarray(im.dataobj)
        if im.shape != ref.shape or not u13_np.allclose(im.affine, ref.affine):
            raise ValueError('ROI must match the analysis grid; supply the required ANTs transforms')
        if not u13_np.isin(a, [0, 1]).all() or not a.any():
            raise ValueError('Mapped ROI must be nonempty and binary')
        u13_nib.save(im, destination)
    return u13_nib.load(destination)


In [ ]:
def u13_roi_define():
    table = u15_table
    file_ref = u15_file_ref
    input_path = u15_input_path
    src = table('ROI_masks.csv', ['roi_id', 'path', 'source_doi', 'independent_of_sample'], ['roi_id'])
    if set(src.roi_id) != set(u13_ROIS):
        raise ValueError('Need the five documented independent ROI masks')
    if not src.independent_of_sample.astype(str).str.lower().isin(['true', '1']).all():
        raise ValueError('ROI independence must be explicitly documented')
    reference = u13_first_path(u13_subjects()[0], 'stroop', 'effect')
    u13_NEW.mkdir(parents=True, exist_ok=True)
    manifest = {}
    rows = []
    cfg = {'source_doi': src.source_doi.tolist(), 'input_manifest_sha256': u13_sha(input_path('ROI_masks.csv')), 'target_space': 'MNI152NLin2009cAsym'}
    u13_atomic_json(u13_ROI_CONFIG, cfg)
    for row in src.itertuples():
        source = file_ref(row.path)
        raw_transforms = getattr(row, 'transforms', None)
        transforms = [] if u13_pd.isna(raw_transforms) or not str(raw_transforms).strip() else u13_json.loads(raw_transforms)
        if not isinstance(transforms, list) or not all((isinstance(p, str) for p in transforms)):
            raise ValueError('transforms must be a JSON array of paths in ANTs CLI order')
        transforms = [file_ref(p) for p in transforms]
        dest = u13_NEW / f'{row.roi_id}_ALE_3D.nii.gz'
        im = u13_roi_prepare_roi(source, reference, dest, transforms)
        a = u13_np.asarray(im.dataobj)
        xyz = u13_nib.affines.apply_affine(im.affine, u13_np.argwhere(a > 0)).mean(axis=0)
        rows.append(dict(roi_id=row.roi_id, target_voxels=int(a.sum()), volume_mm3=float(a.sum() * abs(u13_np.linalg.det(im.affine[:3, :3]))), centroid_x=xyz[0], centroid_y=xyz[1], centroid_z=xyz[2], source=str(source), source_sha256=u13_sha(source), reference_doi=row.source_doi))
        manifest[row.roi_id] = {'sha256': u13_sha(dest), 'source_sha256': u13_sha(source), 'reference_sha256': u13_sha(reference), 'transforms_cli_order': [{'path': str(p), 'sha256': u13_sha(p)} for p in transforms], 'interpolation': 'GenericLabel' if transforms else 'already_on_target_grid'}
    u13_atomic_json(u13_NEW / 'definition_and_provenance.json', {'config_sha256': u13_sha(u13_ROI_CONFIG), 'masks': manifest, 'selection_independent_of_current_sample': True})
    u13_save_table(u13_pd.DataFrame(rows), u13_TASK, 'independent_ALE_3D_ROI_definition_5_regions')
    return u13_pd.DataFrame(rows)


In [ ]:
def u13_roi_random_intercept_ml(y, design):
    beta = u13_np.linalg.lstsq(design, y.mean(axis=0), rcond=None)[0]
    resid = y - design @ beta
    (n, cells) = y.shape

    def loss(logv):
        (noise, intercept) = u13_np.exp(logv)
        ld = (cells - 1) * u13_np.log(noise) + u13_np.log(noise + cells * intercept)
        q = u13_np.sum(resid ** 2) / noise - intercept / (noise * (noise + cells * intercept)) * u13_np.sum(resid.sum(axis=1) ** 2)
        return 0.5 * (n * cells * u13_np.log(2 * u13_np.pi) + n * ld + q)
    fit = u13_minimize(loss, u13_np.log([max(u13_np.var(resid), 0.0001), max(u13_np.var(resid.mean(axis=1)), 0.0001)]), method='L-BFGS-B', bounds=[(-25, 25), (-25, 25)])
    if not fit.success:
        raise RuntimeError(f'Maximum-likelihood fit failed to converge: {fit.message}')
    return (-fit.fun, u13_np.exp(fit.x))


In [ ]:
def u13_roi_activation():
    for radius in u13_ROI_VARIANTS:
        (masks, _) = u13_roi_masks(radius)
        rows = []
        for sid in u13_subjects():
            for task in u13_CONTRAST:
                (t, valid, _, _) = u13_t_map(sid, task)
                for (roi, mask) in masks.items():
                    ok = mask & valid
                    coverage = ok.sum() / mask.sum()
                    eligible = coverage >= 0.8
                    rows.append(dict(subject_id=sid, task=task, roi_id=roi, roi_definition=radius, coverage=float(coverage), eligible_80pct=eligible, mean_voxel_t=float(t[ok].mean()) if eligible else u13_np.nan))
        df = u13_pd.DataFrame(rows)
        u13_NEW.mkdir(parents=True, exist_ok=True)
        df.to_csv(u13_NEW / f'independent_ALE_ROI_{radius}_mean_t.csv', index=False)
        one = []
        for ((task, roi), g) in df.groupby(['task', 'roi_id']):
            one.append(dict(task=task, roi_id=roi, **u13_mean_stats(g.mean_voxel_t)))
        one = u13_pd.DataFrame(one)
        for p in ('p_flip',):
            one[p + '_FDR_10'] = u13_fdr(one[p])
        u13_save_table(one, u13_TASK, f'independent_ALE_ROI_{radius}_10_activation_tests')
        clean = df.dropna(subset=['mean_voxel_t'])
        cells = [(t, r) for t in u13_CONTRAST for r in u13_ROIS]
        y = df.pivot(index='subject_id', columns=['task', 'roi_id'], values='mean_voxel_t').reindex(columns=cells)
        if not y.isna().any().any():
            task = u13_np.repeat([0.0, 1.0], 5)
            roi = u13_np.tile(u13_np.eye(5)[:, 1:], (2, 1))
            reduced = u13_np.column_stack([u13_np.ones(10), task, roi])
            full = u13_np.column_stack([reduced, roi * task[:, None]])
            (llf, vf) = u13_roi_random_intercept_ml(y.to_numpy(), full)
            (llr, vr) = u13_roi_random_intercept_ml(y.to_numpy(), reduced)
            details = dict(full_residual_variance=vf[0], full_intercept_variance=vf[1])
        else:
            import statsmodels.formula.api as smf
            full = smf.mixedlm('mean_voxel_t ~ C(task)*C(roi_id)', clean, groups=clean.subject_id)
            reduced = smf.mixedlm('mean_voxel_t ~ C(task)+C(roi_id)', clean, groups=clean.subject_id)
            fitf = full.fit(reml=False, method=['lbfgs', 'powell'])
            fitr = reduced.fit(reml=False, method=['lbfgs', 'powell'])
            if not fitf.converged or not fitr.converged:
                raise RuntimeError('Unbalanced mixed-effects model failed to converge')
            (llf, llr) = (fitf.llf, fitr.llf)
            details = {}
        lr = max(0.0, 2 * (llf - llr))
        u13_save_table(u13_pd.DataFrame([dict(N_subjects=clean.subject_id.nunique(), N_observations=len(clean), method='ML random-intercept likelihood ratio', LR_chi2=lr, df=4, p=u13_stats.chi2.sf(lr, 4), **details)]), u13_TASK, f'independent_ALE_ROI_{radius}_ROI_by_task_interaction')
        paired = []
        for roi in u13_ROIS:
            p = df[df.roi_id == roi].pivot(index='subject_id', columns='task', values='mean_voxel_t').dropna()
            d = (p.gonogo - p.stroop).to_numpy()
            n = len(d)
            sd = d.std(ddof=1)
            if n < 3 or not u13_np.isfinite(sd) or sd <= 0:
                raise ValueError('Paired ROI inference requires at least three variable paired differences')
            t = d.mean() / u13_stats.sem(d)
            ptost = max(u13_stats.nct.sf(t, n - 1, -0.3 * u13_np.sqrt(n)), u13_stats.nct.cdf(t, n - 1, 0.3 * u13_np.sqrt(n)))
            ci = u13_stats.t.interval(0.95, n - 1, loc=d.mean(), scale=u13_stats.sem(d))
            limit = abs(t) + 100
            dz_ci = [u13_brentq(lambda ncp: u13_stats.nct.cdf(abs(t), n - 1, ncp) - tail, -limit, limit) / u13_np.sqrt(n) for tail in (0.975, 0.025)]
            if t < 0:
                dz_ci = [-dz_ci[1], -dz_ci[0]]
            paired.append(dict(roi_id=roi, N=n, direction='GoNoGo minus Stroop', mean_difference=d.mean(), dz=d.mean() / sd, t=t, df=n - 1, p=2 * u13_stats.t.sf(abs(t), n - 1), CI95_low=ci[0], CI95_high=ci[1], dz_CI95_low=dz_ci[0], dz_CI95_high=dz_ci[1], equivalence_bound_dz=0.3, p_TOST=ptost))
        paired = u13_pd.DataFrame(paired)
        paired['p_FDR_5'] = u13_fdr(paired.p)
        paired['p_TOST_FDR_5'] = u13_fdr(paired.p_TOST)
        u13_save_table(paired, u13_TASK, f'independent_ALE_ROI_{radius}_cross_task_paired_tests')
        print('Independent ALE ROI activation analysis complete', radius, flush=True)


In [ ]:
def u13_roi_fit_unsmoothed(subject, task, paths, mask_img):
    effects = []
    variances = []
    dofs = []
    for p in paths:
        (tr, ref) = u12_glm_acquisition(p['metadata'])
        (confounds, _) = u12_glm_load_confounds(p['confounds'])
        events = u12_glm_reconstruct_events(subject, task, None) if task == 'stroop' else u12_glm_load_events(p['events'], task)
        model = u13_FirstLevelModel(t_r=tr, slice_time_ref=ref, hrf_model='glover + derivative', drift_model='cosine', high_pass=1 / 128, noise_model='ar1', smoothing_fwhm=None, mask_img=mask_img, signal_scaling=0, minimize_memory=True, n_jobs=1, verbose=0).fit(str(p['bold']), events=events, confounds=confounds)
        design = model.design_matrices_[0]
        rank = u13_np.linalg.matrix_rank(design)
        if rank != design.shape[1]:
            raise ValueError('Unsmoothed GLM design matrix is rank deficient')
        dofs.append(int(len(design) - rank))
        con = 'incongruent - congruent' if task == 'stroop' else 'nogo_correct - go_correct'
        effects.append(model.compute_contrast(con, output_type='effect_size'))
        variances.append(model.compute_contrast(con, output_type='effect_variance'))
    if len(paths) == 2:
        (e, v, _, _) = u13_compute_fixed_effects(effects, variances, mask=mask_img, precision_weighted=True, dofs=dofs)
    else:
        (e, v) = (effects[0], variances[0])
    (ed, vd) = (u13_np.asarray(e.dataobj), u13_np.asarray(v.dataobj))
    return u13_np.divide(ed, u13_np.sqrt(u13_np.maximum(vd, 0)), out=u13_np.zeros_like(ed), where=vd > 0)


In [ ]:
def u13_roi_fit_unsmoothed_arrays(subject, task, paths, series):
    effects = []
    variances = []
    for (p, y) in zip(paths, series):
        (tr, ref) = u12_glm_acquisition(p['metadata'])
        (cf, _) = u12_glm_load_confounds(p['confounds'])
        ev = u12_glm_reconstruct_events(subject, task, None) if task == 'stroop' else u12_glm_load_events(p['events'], task)
        frames = u13_np.linspace(ref * tr, (len(y) - 1 + ref) * tr, len(y))
        dm = u13_make_first_level_design_matrix(frames, events=ev, hrf_model='glover + derivative', drift_model='cosine', high_pass=1 / 128, add_regs=cf.to_numpy(), add_reg_names=cf.columns.tolist())
        if u13_np.linalg.matrix_rank(dm) != dm.shape[1]:
            raise ValueError('Unsmoothed GLM design matrix is rank deficient')
        (scaled, _) = u13_mean_scaling(y, axis=0)
        (labels, results) = u13_run_glm(scaled, dm.to_numpy(), noise_model='ar1', bins=100, n_jobs=1)
        contrast = u13_np.zeros(dm.shape[1])
        (positive, negative) = ('incongruent', 'congruent') if task == 'stroop' else ('nogo_correct', 'go_correct')
        contrast[dm.columns.get_loc(positive)] = 1
        contrast[dm.columns.get_loc(negative)] = -1
        fitted = u13_compute_contrast(labels, results, contrast, stat_type='t')
        effects.append(fitted.effect)
        variances.append(fitted.variance)
    if len(effects) == 1:
        (e, v) = (effects[0], variances[0])
    else:
        weights = 1 / u13_np.maximum(u13_np.array(variances), 1e-16)
        v = 1 / weights.sum(axis=0)
        e = (u13_np.array(effects) * weights).sum(axis=0) * v
    return e / u13_np.sqrt(v)


In [ ]:
def u13_roi_extract_roi_series(path, mask):
    img = u13_nib.load(str(path), keep_file_open=True)
    result = u13_np.empty((img.shape[-1], int(mask.sum())), dtype=u13_np.float32)
    for volume in range(img.shape[-1]):
        result[volume] = u13_np.asarray(img.dataobj[..., volume])[mask]
    return result


In [ ]:
def u13_roi_verify_unsmoothed_implementation():
    sid = u13_subjects()[0]
    (masks, ref) = u13_roi_masks()
    union = u13_np.logical_or.reduce([masks[r] for r in u13_ROIS[:3]])
    paths = {'stroop': [u12_glm_paths_for(sid, 'stroop', None)], 'gonogo': [u12_glm_paths_for(sid, 'gonogo', i) for i in ('01', '02')]}
    valid = union.copy()
    for runs in paths.values():
        for p in runs:
            valid &= u13_np.asarray(u13_nib.load(p['mask']).dataobj) > 0
    good = u13_np.ones(valid.sum(), bool)
    series = {task: [] for task in paths}
    for (task, runs) in paths.items():
        for p in runs:
            y = u13_roi_extract_roi_series(p['bold'], valid)
            good &= u13_np.isfinite(y).all(0) & (y.std(0) > 0) & (abs(y.mean(0)) > 1e-06)
            series[task].append(y)
    valid[valid] = good
    mask = u13_nib.Nifti1Image(valid.astype('uint8'), ref.affine)
    rows = []
    for (task, runs) in paths.items():
        fast = u13_roi_fit_unsmoothed_arrays(sid, task, runs, [x[:, good] for x in series[task]])
        reference = u13_roi_fit_unsmoothed(sid, task, runs, mask)[valid]
        assert u13_np.allclose(fast, reference, rtol=0.0005, atol=0.0002)
        rows.append(dict(task=task, voxels=len(fast), max_abs_t_difference=float(u13_np.max(abs(fast - reference))), voxel_t_correlation=float(u13_np.corrcoef(fast, reference)[0, 1])))
    u13_atomic_json(u13_NEW / 'unsmoothed_GLM_equivalence_check.json', dict(subject=sid, comparisons=rows, passed=True, implementation_signature=u13_fingerprint([u13_Path(NOTEBOOK_PATH), u13_Path(NOTEBOOK_PATH)]), note='Actual frozen inputs; extracted-voxel implementation versus Nilearn FirstLevelModel.'))
    print('Current unsmoothed GLM numerical-equivalence check passed', flush=True)


In [ ]:
def u13_roi_patterns():
    sets = {r: u13_roi_masks(r) for r in u13_ROI_VARIANTS}
    ref = sets['ALE_3D'][1]
    union = u13_np.logical_or.reduce([m for r in sets for (k, m) in sets[r][0].items() if k in u13_ROIS[:3]])
    cache = u13_NEW
    cache.mkdir(parents=True, exist_ok=True)
    allrows = []
    from concurrent.futures import ThreadPoolExecutor as ThreadPoolExecutor
    from threadpoolctl import threadpool_limits as threadpool_limits

    def work(sid):
        paths = {'stroop': [u12_glm_paths_for(sid, 'stroop', None)], 'gonogo': [u12_glm_paths_for(sid, 'gonogo', r) for r in ('01', '02')]}
        inputs = [p[k] for runs in paths.values() for p in runs for k in ('bold', 'mask', 'confounds', 'metadata')]
        inputs += [paths['gonogo'][j]['events'] for j in range(2)] + [u12_glm_raw_log_path(sid, 'stroop', None), u13_ROI_CONFIG, u13_Path(NOTEBOOK_PATH), u13_Path(NOTEBOOK_PATH)]
        fp = u13_fingerprint(inputs, dict(version=u13_VERSION, nilearn=__import__('nilearn').__version__))
        target = cache / f'{sid}_unsmoothed_patterns.json'
        if target.exists() and u13_json.loads(target.read_text()).get('fingerprint') == fp:
            return u13_json.loads(target.read_text())['rows']
        valid = union.copy()
        for runs in paths.values():
            for p in runs:
                im = u13_nib.load(p['mask'])
                if im.shape != ref.shape or not u13_np.allclose(im.affine, ref.affine):
                    raise ValueError('ROI and BOLD spaces do not match')
                valid &= u13_np.asarray(im.dataobj) > 0
        initial = u13_nib.Nifti1Image(valid.astype('uint8'), ref.affine)
        good = u13_np.ones(int(valid.sum()), bool)
        series = {task: [] for task in paths}
        for (task, runs) in paths.items():
            for p in runs:
                ts = u13_roi_extract_roi_series(p['bold'], valid)
                good &= u13_np.isfinite(ts).all(axis=0) & (ts.std(axis=0) > 0) & (abs(ts.mean(axis=0)) > 1e-06)
                series[task].append(ts)
        valid[valid] = good
        if not valid.any():
            raise ValueError(f'{sid}:Shared valid mask is empty after unsmoothing')
        mask = u13_nib.Nifti1Image(valid.astype('uint8'), ref.affine)
        t = {task: u13_roi_fit_unsmoothed_arrays(sid, task, runs, [ts[:, good] for ts in series[task]]) for (task, runs) in paths.items()}
        rows = []
        for (radius, (masks, _)) in sets.items():
            for roi in u13_ROIS[:3]:
                m = masks[roi] & valid
                coverage = m.sum() / masks[roi].sum()
                r = z = None
                if coverage >= 0.8:
                    (x, y) = (t['stroop'][m[valid]], t['gonogo'][m[valid]])
                    if len(x) < 3 or x.std() == 0 or y.std() == 0:
                        raise ValueError('Pattern has zero variance or too few voxels')
                    r = float(u13_np.corrcoef(x, y)[0, 1])
                    z = float(u13_np.arctanh(u13_np.clip(r, -1 + 1e-12, 1 - 1e-12)))
                rows.append(dict(subject_id=sid, roi_definition=radius, roi_id=roi, coverage=float(coverage), eligible_80pct=coverage >= 0.8, voxels=int(m.sum()), pattern_r=r, fisher_z=z))
        u13_atomic_json(target, dict(fingerprint=fp, rows=rows))
        print('Unsmoothed-pattern analysis complete', sid, flush=True)
        return rows
    with threadpool_limits(limits=1), ThreadPoolExecutor(max_workers=3) as pool:
        for rows in pool.map(work, u13_subjects()):
            allrows.extend(rows)
    df = u13_pd.DataFrame(allrows)
    df.to_csv(u13_NEW / 'independent_ALE_ROI_unsmoothed_patterns.csv', index=False)
    for radius in u13_ROI_VARIANTS:
        summary = []
        for (roi, g) in df[df.roi_definition == radius].groupby('roi_id'):
            summary.append(dict(roi_id=roi, **u13_mean_stats(g.fisher_z, alternative='greater')))
        out = u13_pd.DataFrame(summary)
        out['p_positive_FDR_3'] = u13_fdr(out.p_flip_positive)
        u13_save_table(out, u13_TASK, f'independent_ALE_ROI_{radius}_pattern_similarity_one_sided')


In [ ]:
import matplotlib.pyplot as u13_plt
from matplotlib.text import Text as u13_Text
from matplotlib.ticker import FixedLocator as u13_FixedLocator, FixedFormatter as u13_FixedFormatter


In [ ]:
def u13_apa_plain_apa(text, bold=False):
    task_pattern = 'Go/No-Go|GoNoGo|Stroop|GNG(?: [12])?'

    def fragment(part):

        def task_math(match):
            cmd = 'mathbfit' if bold else 'mathit'
            word = match[0].replace(' ', chr(92) + ' ')
            pieces = word.split('-')
            return '$' + (chr(92) + 'text{-}').join((chr(92) + cmd + '{' + piece + '}' for piece in pieces)) + '$'
        part = u13_re.sub(task_pattern, task_math, part)
        tokens = u13_re.split('(\\$[^$]*\\$)', part)
        for (i, token) in enumerate(tokens):
            if not token.startswith('$'):
                tokens[i] = u13_re.sub('\\b(r|p|t|z|Z|N|n|M|SD|SE|df|d|F|v|a)\\b', lambda m: '$' + m[0] + '$', token)
        return ''.join(tokens)
    pieces = u13_re.split('(\\$[^$]*\\$)', text)
    return ''.join((part if part.startswith('$') else fragment(part) for part in pieces))


In [ ]:
def u13_apa_apply_apa_style(fig):
    if getattr(fig, '_a8_APA_applied', False):
        return
    fig._a8_APA_applied = True
    for axis in [axis for ax in fig.axes for axis in [ax.xaxis, ax.yaxis]]:
        current = [v.get_text() for v in axis.get_ticklabels()]
        changed = [u13_apa_plain_apa(v) for v in current]
        if current != changed:
            axis.set_major_locator(u13_FixedLocator(axis.get_majorticklocs()))
            axis.set_major_formatter(u13_FixedFormatter(changed))
    for text in fig.findobj(match=u13_Text):
        value = text.get_text()
        if not value:
            continue
        out = u13_apa_plain_apa(value, bold=text.get_fontweight() in ['bold', 'heavy', 700, 800, 900])
        if out != value:
            text.set_text(out)


In [ ]:
import re as u13_re
import numpy as u13_np
import pandas as u13_pd
from matplotlib.transforms import Bbox as u13_Bbox


In [ ]:
def u13_style_apply_final_style(fig, root, name):
    if getattr(fig, '_a8_final_style_applied', False):
        return
    fig._a8_final_style_applied = True
    ax = fig.axes[0]
    if name == 'Figure_S1_representative_design_matrix_n106':
        fig._a8_tight_export = True
    elif name == 'Figure_S1_head_motion_distribution_n106':
        for c in ax.collections:
            c.set_alpha(0.9)
            c.set_edgecolors('black')
            c.set_linewidths(0.25)
    elif name.startswith('Figure_S4_'):
        top = 304 if 'Stroop' in name else 219
        fig._a8_export_bbox = u13_Bbox.from_extents(150 / 600, 38 / 600, 4191 / 600, (2100 - top) / 600)
    elif name.startswith('Figure_S2_'):
        if 'individual_Dice_distribution' in name:
            for c in ax.collections:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        elif 'activation_counts' in name:
            fig.legends[0].set_bbox_to_anchor((0.55, 0.06), transform=fig.transFigure)
    elif name.startswith('classical_brain_behavior_'):
        fig.set_size_inches(2.9, 1.9)
        rest = 'rest_connectivity' in name
        fig.subplots_adjust(left=0.37 if rest else 0.25, right=0.98, bottom=0.3, top=0.98)
        ax.xaxis.label.set_fontsize(6.5)
        ax.tick_params(labelsize=5.5)
        legend = fig.legends[0]
        legend.set_bbox_to_anchor((0.55, 0.01), transform=fig.transFigure)
        for text in legend.get_texts():
            text.set_fontsize(5.5)
        if rest:
            data = u13_pd.read_excel(u13_style_ROOT / '1.resting_state/tables/brain_behavior_4_connectivity_by_2_behavior_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            key = 'composite'
            order = ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']
        else:
            data = u13_pd.read_excel(u13_style_ROOT / '2.task_fmri/tables/brain_behavior_5ROI_by_2task_by_2behavior_ALE_3D.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            data = data[data.task.eq('stroop' if 'Stroop' in name else 'gonogo')]
            key = 'roi_id'
            order = u13_ROIS
        for (k, outcome) in enumerate(['stroop_rt_cost_ms', 'nogo_accuracy']):
            for (j, (_, row)) in enumerate(data[data.outcome.eq(outcome)].set_index(key).loc[order].iterrows()):
                xpos = float(row.CI95_high) + 0.025
                if xpos < 0 and xpos + 0.15 > 0:
                    xpos = 0.025
                ax.text(xpos, j + (k - 0.5) * 0.27, f'{float(row.r):.3f}', ha='left', va='center', fontsize=5.2, color=['#3C5488', '#E64B35'][k])
    elif name.startswith('Figure_S7_'):
        fig.set_size_inches(3.6, 2.1)
        fig.subplots_adjust(left=0.2, right=0.985, bottom=0.2, top=0.89)
        ax.set_ylabel('Cross-task pattern\nsimilarity (Fisher z)', fontsize=7, labelpad=3)
        ax.tick_params(labelsize=6)
        for c in ax.collections:
            if len(c.get_offsets()) > 100:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        data = u13_pd.read_excel(u13_style_ROOT / '2.task_fmri/tables/independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
        for (label, key) in zip(ax.texts, u13_ROIS[:3]):
            val = float(data.loc[key, 'p_positive_FDR_3'])
            label.set_text('$p_{\\mathrm{FDR}} = ' + f'{val:.3f}' + '$')
            label.set_y(1.015)
            label.set_fontsize(7.5)
    elif name in ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']:
        base = u13_style_ROOT / '2.task_fmri/tables'
        if name == 'Figure_independent_ALE_ROI_activation_n106':
            one = u13_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            for (k, task) in enumerate(u13_CONTRAST):
                for (j, (_, row)) in enumerate(one[one.task.eq(task)].set_index('roi_id').loc[u13_ROIS].iterrows()):
                    if row.p_flip_FDR_10 < 0.05:
                        ax.text(float(row.CI95_high) + 0.025, j + (k - 0.5) * 0.22, '*', fontsize=8, color=['#397A9E', '#B76B3F'][k], ha='left', va='center')
            (lo, hi) = ax.get_xlim()
            ax.set_xlim(lo, hi + 0.06)
        else:
            paired = 'paired' in name
            data = u13_pd.read_excel(base / ('independent_ALE_ROI_ALE_3D_cross_task_paired_tests.xlsx' if paired else 'independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
            for (j, key) in enumerate(u13_ROIS if paired else u13_ROIS[:3]):
                q = float(data.loc[key, 'p_FDR_5' if paired else 'p_positive_FDR_3'])
                value = '< 0.001' if q < 0.001 else '= ' + (f'{q:.5f}' if 0.05 < q < 0.0505 else f'{q:.3f}')
                ax.text(0.98, j - 0.26, '$p_{\\mathrm{FDR}} ' + value + '$', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=7.5)
    elif name in ['Figure_independent_ALE_ROI_resting_connectivity_matrix_n106', 'Figure_independent_ALE_ROI_four_prespecified_connectivity_measures_n106']:
        for text in list(fig.texts):
            text.remove()
        fig._a8_tight_export = True
        if 'matrix' in name:
            fig.set_size_inches(2.8, 2.5)
            fig.subplots_adjust(left=0.21, right=0.81, bottom=0.25, top=0.93)
            fig.axes[1].set_position([0.84, 0.25, 0.025, 0.68])
            fig.axes[1].yaxis.label.set_fontsize(6.5)
            fig.axes[1].tick_params(labelsize=5.5)
            edges = u13_pd.read_excel(u13_style_ROOT / '1.resting_state/tables/functional_connectivity_10_edges_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            lookup = {frozenset(row.edge.split('__')): row for (_, row) in edges.iterrows()}
            for text in ax.texts:
                (j, i) = map(int, text.get_position())
                row = lookup[frozenset([u13_ROIS[i], u13_ROIS[j]])]
                text.set_text(f"{float(row['mean']):.2f}" + ('*' if row.p_flip_FDR_10 < 0.05 else ''))
                text.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)
        else:
            fig.set_size_inches(2.9, 1.9)
            fig.subplots_adjust(left=0.38, right=0.98, bottom=0.23, top=0.98)
            ax.xaxis.label.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)


In [ ]:
import os as u13_os
import subprocess as u13_subprocess
import matplotlib as u13_matplotlib
u13_matplotlib.use('Agg')
import matplotlib.pyplot as u13_plt
from matplotlib.lines import Line2D as u13_Line2D
import numpy as u13_np
import pandas as u13_pd
u13_PLOT_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u13_PLOT_COLORS = {'stroop': '#397A9E', 'gonogo': '#B76B3F'}
u13_PLOT_TASK_FIGURE_NAMES = ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']
u13_PLOT_QC_FIGURE_NAMES = ['Figure_S1_representative_design_matrix_n106', 'Figure_S1_head_motion_distribution_n106']
u13_PLOT_COVERAGE_FIGURE_NAMES = ['Figure_S1_Stroop_standard_space_brain_coverage_n106', 'Figure_S1_GoNoGo_standard_space_brain_coverage_n106']
u13_PLOT_MAXT_FIGURE_NAMES = ['Figure_S4_Stroop_voxelwise_maxT_FWE_n106', 'Figure_S4_GoNoGo_voxelwise_maxT_FWE_n106']


In [ ]:
def u13_export(fig, root, name):
    u13_style_apply_final_style(fig, root, name)
    u13_apa_apply_apa_style(fig)
    out = root / 'image'
    out.mkdir(exist_ok=True)
    fig.canvas.draw()
    fig.savefig(out / (name + '.png'), dpi=600, facecolor='white', bbox_inches='tight')
    u13_plt.close(fig)


In [ ]:
def u13_plot_task(figure_names=None):
    base = u13_TASK / 'tables'
    one = u13_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    paired = u13_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_cross_task_paired_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id').loc[u13_ROIS]
    pat = u13_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id').loc[u13_ROIS[:3]]
    for (index, name) in enumerate(u13_PLOT_TASK_FIGURE_NAMES):
        if figure_names is not None and name not in figure_names:
            continue
        (fig, ax) = u13_plt.subplots(figsize=[(3.3, 2.6), (3.3, 2.3), (3.3, 2.0)][index])
        fig._a8_tight_export = True
        if index == 2:
            fig.subplots_adjust(left=0.2, right=0.98, bottom=0.25, top=0.98)
        else:
            fig.subplots_adjust(left=0.24, right=0.98, bottom=0.25 if index == 0 else 0.2, top=0.98)
        if index == 0:
            for (k, t) in enumerate(u13_CONTRAST):
                d = one[one.task == t].set_index('roi_id').loc[u13_ROIS]
                y = u13_np.arange(5) + (k - 0.5) * 0.22
                ax.errorbar(d['mean'], y, xerr=u13_np.array([d['mean'] - d.CI95_low, d.CI95_high - d['mean']]), fmt='o', color=u13_PLOT_COLORS[t], ms=3.2, lw=0.9, capsize=2)
            ax.set(yticks=u13_np.arange(5), yticklabels=u13_PLOT_LABELS, ylim=(4.6, -0.6), xlabel='Mean voxel t')
            fig.legend([u13_Line2D([0], [0], marker='o', color=u13_PLOT_COLORS[t], lw=0.8, ms=3) for t in u13_CONTRAST], ['Stroop', 'Go/No-Go'], loc='lower center', bbox_to_anchor=(0.6, 0.01), ncol=2, fontsize=7)
        elif index == 1:
            ax.errorbar(paired.mean_difference, u13_np.arange(5), xerr=u13_np.array([paired.mean_difference - paired.CI95_low, paired.CI95_high - paired.mean_difference]), fmt='o', color='#53636B', ms=3.2, lw=0.9, capsize=2)
            ax.set(yticks=u13_np.arange(5), yticklabels=u13_PLOT_LABELS, ylim=(4.6, -0.6), xlabel='Go/No-Go minus Stroop')
        else:
            ax.errorbar(pat['mean'], u13_np.arange(3), xerr=u13_np.array([pat['mean'] - pat.CI95_low, pat.CI95_high - pat['mean']]), fmt='o', color='#725A8E', ms=3.2, lw=0.9, capsize=2)
            ax.set(yticks=u13_np.arange(3), yticklabels=u13_PLOT_LABELS[:3], ylim=(2.6, -0.6), xlabel='Pattern Fisher z')
        ax.axvline(0, color='.7', lw=0.6, ls='--', zorder=0)
        u13_export(fig, u13_TASK, name)


In [ ]:
import numpy as u13_np
import pandas as u13_pd
import matplotlib.pyplot as u13_plt
from matplotlib.lines import Line2D as u13_Line2D
from matplotlib.patches import Patch as u13_Patch
u13_SUPP_DICE_NAMES = ['Figure_S2_p001_individual_Dice_distribution_n106', 'Figure_S2_p001_activation_counts_n106', 'Figure_S2_p01_individual_Dice_distribution_n106', 'Figure_S2_p01_activation_counts_n106']
u13_SUPP_BEHAVIOR_NAMES = ['Supplementary_classical_brain_behavior_Stroop_activation_partial_correlations_n106', 'Supplementary_classical_brain_behavior_GoNoGo_activation_partial_correlations_n106', 'Supplementary_classical_brain_behavior_rest_connectivity_partial_correlations_n106']
u13_SUPP_PATTERN_NAME = 'Figure_S7_shared_ROI_cross_task_pattern_similarity_individual_distribution_n106'
u13_SUPP_REGION_LABELS = ['Whole brain', 'SMA', 'rAI/FO', 'lAI']
u13_SUPP_ROI_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u13_SUPP_OUTCOMES = ['stroop_rt_cost_ms', 'nogo_accuracy']
u13_SUPP_BEHAVIOR_COLORS = ['#3C5488', '#E64B35']
u13_SUPP_STATUS_COLORS = ['#3C5488', '#AAB8C3', '#E8ECEF', '#F39B7F']


In [ ]:
def u13_dice_distribution(figure_names=None):
    path = u13_TASK / 'tables/individual_Dice_whole_brain_and_shared_ROIs_subject_distribution.xlsx'
    table = u13_pd.read_excel(path, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    assert not table.duplicated(['subject_id', 'region', 'p_uncorrected']).any()
    cohort_n = table.subject_id.nunique()
    assert table.direction.eq('two_sided').all()
    provenance = []
    rendered_thresholds = 0
    for (col, p) in enumerate([0.001, 0.01]):
        if figure_names is not None and (not any((n in figure_names for n in u13_SUPP_DICE_NAMES[col * 2:col * 2 + 2]))):
            continue
        rendered_thresholds += 1
        (dist_fig, top) = u13_plt.subplots(figsize=(3.2, 1.9))
        dist_fig.subplots_adjust(left=0.38, right=0.97, bottom=0.2, top=0.98)
        (count_fig, bottom) = u13_plt.subplots(figsize=(3.8, 2.35))
        count_fig.subplots_adjust(left=0.22, right=0.97, bottom=0.31, top=0.98)
        panel_rows = []
        ylabels = []
        for (row, (region, label)) in enumerate(zip(u13_SUPP_REGIONS, u13_SUPP_REGION_LABELS)):
            sub = table[table.region.eq(region) & u13_np.isclose(table.p_uncorrected, p)].sort_values('subject_id')
            assert sub.subject_id.is_unique
            eligible = sub.eligible_80pct.to_numpy(bool)
            neither = sub.neither_active.to_numpy(bool)
            overlap = sub.overlap.to_numpy(bool)
            defined = sub.loc[eligible & sub.dice.notna().to_numpy(), 'dice']
            assert u13_np.array_equal(sub.dice.notna().to_numpy(), eligible & ~neither)
            assert defined.between(0, 1).all()
            offsets = u13_np.linspace(-0.15, 0.15, len(defined))
            top.scatter(defined.to_numpy(), row + offsets, s=4, color='#4DBBD5', alpha=0.65, linewidths=0, rasterized=True, zorder=2)
            (q25, median, q75) = defined.quantile([0.25, 0.5, 0.75]).to_numpy()
            top.plot([q25, q75], [row, row], color='#222222', lw=1.2, zorder=3)
            top.plot([median, median], [row - 0.12, row + 0.12], color='#222222', lw=1.1, zorder=3)
            ylabels.append(f'{label} ({len(defined)}/{eligible.sum()})')
            states = [eligible & overlap, eligible & ~overlap & ~neither, eligible & neither, ~eligible]
            counts = [int(x.sum()) for x in states]
            assert sum(counts) == len(sub) and counts[0] + counts[1] == len(defined)
            start = 0
            for (index, (count, color)) in enumerate(zip(counts, u13_SUPP_STATUS_COLORS)):
                bottom.barh(row, count, left=start, height=0.53, color=color, edgecolor='white', linewidth=0.4, zorder=2)
                if count >= 9:
                    bottom.text(start + count / 2, row, str(count), ha='center', va='center', fontsize=6, color='white' if index == 0 else '#222222')
                start += count
            panel_rows.append(dict(region=region, p_uncorrected=p, n_total=len(sub), n_eligible=int(eligible.sum()), n_defined=len(defined), n_overlap=counts[0], n_zero_dice=counts[1], n_both_empty_eligible=counts[2], n_ineligible=counts[3], median=float(median), q25=float(q25), q75=float(q75), displayed_subjects=sub.loc[sub.dice.notna(), 'subject_id'].tolist(), displayed_dice=defined.tolist()))
        top.set(xlim=(-0.025, 1.025), ylim=(3.55, -0.55), yticks=range(4), yticklabels=ylabels, xticks=[0, 0.2, 0.4, 0.6, 0.8], xlabel='Individual Dice')
        bottom.set(xlim=(0, cohort_n), ylim=(3.55, -0.55), yticks=range(4), yticklabels=u13_SUPP_REGION_LABELS, xticks=u13_np.linspace(0, cohort_n, 5), xlabel='Participants')
        for ax in (top, bottom):
            ax.tick_params(labelsize=6, length=2)
        threshold = f'p < {p:g}, uncorrected'
        top.set_xlabel(f'Individual Dice ({threshold})', fontsize=7)
        bottom.set_xlabel(f'Participants ({threshold})', fontsize=7)
        count_fig.legend([u13_Patch(facecolor=c) for c in u13_SUPP_STATUS_COLORS], ['Overlap', 'Active, no overlap', 'Both inactive', 'Coverage < 80%'], loc='lower center', bbox_to_anchor=(0.5, 0.06), ncol=4, fontsize=6, handlelength=1.15, columnspacing=1.2)
        for (fig, name) in [(dist_fig, u13_SUPP_DICE_NAMES[col * 2]), (count_fig, u13_SUPP_DICE_NAMES[col * 2 + 1])]:
            if figure_names is not None and name not in figure_names:
                u13_plt.close(fig)
                continue
            fig._a8_tight_export = True
            u13_export(fig, u13_TASK, name)
            u13_atomic_json(u13_QC / (name + '.source.json'), dict(source_sha256=u13_sha(path), rows=panel_rows, source_rows=len(table), inference='No new inference; existing participant values and disjoint display counts only', missingness='Undefined Dice and low-coverage counts are computed from the supplied subjects; both-empty counts are restricted to eligible participants.'))
        provenance.extend(panel_rows)
    assert len(provenance) == 4 * rendered_thresholds


In [ ]:
def u13_pattern_similarity_distribution():
    from scipy import stats as stats
    NEW = u17_NEW
    individual_path = NEW / 'independent_ALE_ROI_unsmoothed_patterns.csv'
    summary_path = u13_TASK / 'tables/independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx'
    individual = u13_pd.read_csv(individual_path, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    summary = u13_pd.read_excel(summary_path, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    assert individual.roi_definition.eq('ALE_3D').all()
    assert not individual.duplicated(['subject_id', 'roi_id']).any()
    assert set(summary.roi_id) == set(u13_ROIS[:3])
    colours = ['#E64B35', '#4DBBD5', '#00A087']
    (fig, ax) = u13_plt.subplots(figsize=(4.2, 2.5))
    fig.subplots_adjust(left=0.17, right=0.98, bottom=0.2, top=0.87)
    records = []
    for (i, (key, label, colour)) in enumerate(zip(u13_ROIS[:3], u13_SUPP_ROI_LABELS[:3], colours)):
        sub = individual[individual.roi_id.eq(key)].sort_values('subject_id')
        assert sub.subject_id.is_unique
        good = sub[sub.eligible_80pct].copy()
        bad = sub[~sub.eligible_80pct]
        assert good.coverage.ge(0.8).all() and bad.fisher_z.isna().all()
        values = good.fisher_z.to_numpy(float)
        assert u13_np.isfinite(values).all()
        assert u13_np.allclose(u13_np.tanh(values), good.pattern_r, atol=1e-12)
        row = summary[summary.roi_id.eq(key)].iloc[0]
        assert len(values) == int(row.N)
        assert u13_np.isclose(values.mean(), row['mean'], atol=1e-12)
        assert u13_np.isclose(values.std(ddof=1), row.SD, atol=1e-12)
        interval = stats.t.interval(0.95, len(values) - 1, loc=values.mean(), scale=stats.sem(values))
        assert u13_np.allclose(interval, [row.CI95_low, row.CI95_high], atol=1e-12)
        offset = u13_np.linspace(-0.13, 0.13, len(values))
        ax.scatter(i + offset, values, s=10, color=colour, alpha=0.55, edgecolors='white', linewidths=0.25, rasterized=True, zorder=2)
        ax.errorbar(i + 0.25, row['mean'], yerr=u13_np.array([[row['mean'] - row.CI95_low], [row.CI95_high - row['mean']]]), fmt='D', ms=4, color='#222222', lw=1.2, capsize=3, capthick=0.9, zorder=4)
        ax.text(i, 1.075, f'FDR p = {row.p_positive_FDR_3:.6f}', transform=ax.get_xaxis_transform(), ha='center', fontsize=7)
        records.append(dict(roi_id=key, label=label, N_total=len(sub), N_displayed=len(good), N_coverage_excluded=len(bad), excluded_subjects=bad.subject_id.tolist(), points=good[['subject_id', 'coverage', 'voxels', 'pattern_r', 'fisher_z']].assign(display_x=i + offset).to_dict('records'), mean=float(row['mean']), CI95_low=float(row.CI95_low), CI95_high=float(row.CI95_high), p_positive_FDR_3=float(row.p_positive_FDR_3)))
    all_values = u13_np.concatenate([u13_np.array([p['fisher_z'] for p in r['points']]) for r in records])
    extent = max(abs(all_values.min()), abs(all_values.max())) + 0.09
    ax.axhline(0, color='#737C82', ls='--', lw=0.75, zorder=1)
    ax.set(xlim=(-0.45, 2.65), ylim=(-extent, extent), xticks=range(3), xticklabels=[f"{r['label']}\nn = {r['N_displayed']}" for r in records], ylabel='Cross-task pattern similarity (Fisher z)')
    ax.yaxis.label.set_size(8)
    ax.tick_params(labelsize=7, length=2.5)
    fig._a8_tight_export = True
    u13_export(fig, u13_TASK, u13_SUPP_PATTERN_NAME)
    u13_atomic_json(u13_QC / (u13_SUPP_PATTERN_NAME + '.source.json'), dict(inputs_sha256={p.name: u13_sha(p) for p in [individual_path, summary_path]}, total_input_rows=len(individual), total_points_displayed=sum((r['N_displayed'] for r in records)), rows=records, inference='Existing 50,000-sign-flip inference and FDR values read from table; no new tests', uncertainty='Existing two-sided 95% t intervals for means, not significance-adjusted intervals', point_position='Observed Fisher-z values unchanged; deterministic horizontal separation within ROI only', significant_positive_rois=[r['roi_id'] for r in records if r['p_positive_FDR_3'] < 0.05], interpretation='Significance is determined from the supplied analysis, without an expected direction or outcome'))


In [ ]:
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_file_ref(value):
    if u15_pd.isna(value) or not str(value).strip():
        raise ValueError('Empty input path')
    p = u15_Path(str(value)).expanduser()
    p = p if p.is_absolute() else u15_DATA / p
    if not p.is_file():
        raise FileNotFoundError(p)
    return p.resolve()


In [ ]:
def u15_imaging_manifest():
    d = u15_table('MRI_manifest.csv', ['subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'])
    d['run'] = d.run.fillna('').str.replace('\\.0$', '', regex=True).str.lstrip('0').replace({'': 'single'})
    u15_match_cohort(d, 'MRI_manifest', ('subject_id', 'task', 'run'))
    if not d.task.isin(['stroop', 'gonogo', 'rest']).all():
        raise ValueError('Invalid MRI task')
    return d


In [ ]:
def u15_imaging_paths(subject, task, run=None):
    run = str(run or 'single').lstrip('0') or 'single'
    d = u15_imaging_manifest()
    g = d[(d.subject_id == subject) & (d.task == task) & (d.run == run)]
    if len(g) != 1:
        raise ValueError(f'Expected one MRI run: {subject}/{task}/{run}, found {len(g)}')
    row = g.iloc[0]
    cols = ['bold', 'mask', 'confounds', 'metadata'] + ([] if task == 'rest' else ['events'])
    return {c: u15_file_ref(row[c]) for c in cols}


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
from pathlib import Path as u17__A8Path
from pathlib import Path as u17_Path
import sys as u17_sys
import hashlib as u17_hashlib
import json as u17_json
import os as u17_os
for u17__key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u17_os.environ.setdefault(u17__key, '1')
import nibabel as u17_nib
import numpy as u17_np
import pandas as u17_pd
import nilearn as u17_nilearn
if u17_nilearn.__version__ != '0.13.1':
    raise RuntimeError('Use Nilearn 0.13.1, as specified in the manuscript')
from scipy import stats as u17_stats
from statsmodels.stats.multitest import multipletests as u17_multipletests
import sys as u17_sys
from pathlib import Path as u17_Path
u17_PROJECT = u15_ROOT
u17_RESULTS = u15_RESULTS
u17_DATA = u15_DATA
u17_EXPECTED_N = u15_EXPECTED_N
u17_participants = u15_participants
u17_source_signature = u15_source_signature
u17_MRI = u17_RESULTS / '3.mri_analysis'
u17_TASK = u17_MRI / '2.task_fmri'
u17_REST = u17_MRI / '1.resting_state'
u17_VERSION = 'observed_inputs_v1'
u17_LEGACY = u17_DATA / 'ale_masks'
u17_NEW = u17_TASK / 'other_outputs'
u17_REST_NEW = u17_REST / 'other_outputs'
u17_ROI_CONFIG = u17_MRI / 'ROI_definition_ALE_3D_clusters.json'
u17_ROI_VARIANTS = ('ALE_3D',)
u17_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u17_CONTRAST = {'stroop': 'incongruent_minus_congruent', 'gonogo': 'nogo_correct_minus_go_correct'}
u17_N_PERM = 50000
u17_SEED = 20261017


In [ ]:
def u17_subjects():
    return u17_participants().subject_id.tolist()


In [ ]:
def u17_sha(path):
    h = u17_hashlib.sha256()
    with u17_Path(path).open('rb') as f:
        for block in iter(lambda : f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
def u17_fingerprint(paths, extra=None):
    info = [(str(u17_Path(p).resolve()), u17_Path(p).stat().st_size, u17_Path(p).stat().st_mtime_ns) for p in paths]
    return u17_hashlib.sha256(u17_json.dumps([info, extra, u17_source_signature()], sort_keys=True).encode()).hexdigest()


In [ ]:
def u17_atomic_json(path, data):
    path = u17_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')

    def native(value):
        if isinstance(value, u17_np.generic):
            return value.item()
        if isinstance(value, u17_np.ndarray):
            return value.tolist()
        raise TypeError(f'Unsupported metadata type: {type(value).__name__}')
    tmp.write_text(u17_json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False, default=native))
    tmp.replace(path)


In [ ]:
def u17_save_table(frame, root, name):
    if '10_activation_tests' in name or name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['p_flip_positive'], errors='ignore')
    if name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    if 'pattern_similarity_' in name:
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    dest = root / 'tables'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'{name}.xlsx'
    frame.to_excel(path, index=False)
    from openpyxl import load_workbook as load_workbook
    from openpyxl.styles import Font as Font, Border as Border, Side as Side, Alignment as Alignment
    from openpyxl.utils import get_column_letter as get_column_letter
    wb = load_workbook(path)
    ws = wb.active
    rule = Side(style='thin', color='333333')
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'A2'
    for (col, key) in enumerate(frame.columns, 1):
        letter = get_column_letter(col)
        ws.column_dimensions[letter].width = min(38, max(13, len(str(key)) + 2))
        cell = ws.cell(1, col)
        cell.font = Font(name='Arial', size=10, bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical='center')
        cell.border = Border(top=rule, bottom=rule)
        for row in range(2, ws.max_row + 1):
            c = ws.cell(row, col)
            c.font = Font(name='Arial', size=10)
            if isinstance(c.value, str):
                c.alignment = Alignment(wrap_text=True, vertical='top')
                visual_length = sum((2 if ord(char) > 255 else 1 for char in c.value))
                lines = max(1, int(u17_np.ceil(visual_length / ws.column_dimensions[letter].width)))
                ws.row_dimensions[row].height = max(ws.row_dimensions[row].height or 15, min(150, lines * 15))
            if isinstance(c.value, float):
                c.number_format = '0.000E+00' if str(key).startswith(('p', 'q')) else '0.000'
            if row == ws.max_row:
                c.border = Border(bottom=rule)
    ws.row_dimensions[1].height = 32
    wb.save(path)


In [ ]:
def u17_fdr(p):
    p = u17_np.asarray(p, float)
    out = u17_np.full(p.shape, u17_np.nan)
    valid = u17_np.isfinite(p)
    if valid.any():
        out[valid] = u17_multipletests(p[valid], method='fdr_bh')[1]
    return out


In [ ]:
def u17_mean_stats(x, seed=u17_SEED, alternative='two-sided'):
    if alternative not in ('two-sided', 'greater'):
        raise ValueError('Unsupported alternative')
    key = 'p_flip' if alternative == 'two-sided' else 'p_flip_positive'
    x = u17_np.asarray(x, float)
    x = x[u17_np.isfinite(x)]
    n = len(x)
    if n < 3:
        return dict(N=n, mean=u17_np.nan, SD=u17_np.nan, t=u17_np.nan, df=n - 1, CI95_low=u17_np.nan, CI95_high=u17_np.nan, **{key: u17_np.nan})
    mu = x.mean()
    rng = u17_np.random.default_rng(seed)
    exceed = 0
    for start in range(0, u17_N_PERM, 1000):
        signs = rng.choice((-1.0, 1.0), size=(min(1000, u17_N_PERM - start), n))
        perm = signs @ x / n
        exceed += u17_np.count_nonzero(abs(perm) >= abs(mu) - 1e-14) if alternative == 'two-sided' else u17_np.count_nonzero(perm >= mu - 1e-14)
    sem = u17_stats.sem(x)
    ci = u17_stats.t.interval(0.95, n - 1, loc=mu, scale=sem)
    return dict(N=n, mean=float(mu), SD=float(x.std(ddof=1)), t=float(mu / sem), df=n - 1, CI95_low=float(ci[0]), CI95_high=float(ci[1]), **{key: (exceed + 1) / (u17_N_PERM + 1)})


In [ ]:
def u17_first_path(sid, task, stat):
    name = f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete; fallback to previous input results is prohibited: {sid}/{task}')
    return fresh


In [ ]:
def u17_new_first_complete(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    qc = u17_NEW / f'{sid}_task-{suffix}_first_level_qc.csv'
    maps = [u17_NEW / f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz' for stat in ('effect', 'variance', 'z')]
    maps.append(u17_NEW / f'{sid}_task-{task}_mask-analysis.nii.gz')
    return qc.exists() and all((p.exists() for p in maps))


In [ ]:
def u17_qc_path(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    name = f'{sid}_task-{suffix}_first_level_qc.csv'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete: {sid}/{task}')
    return fresh


In [ ]:
def u17_t_map(sid, task):
    e = u17_nib.load(u17_first_path(sid, task, 'effect'))
    v = u17_nib.load(u17_first_path(sid, task, 'variance'))
    if e.shape != v.shape or not u17_np.allclose(e.affine, v.affine):
        raise ValueError('Effect and variance image spaces do not match')
    (ed, vd) = (u17_np.asarray(e.dataobj, float), u17_np.asarray(v.dataobj, float))
    valid = u17_np.isfinite(ed) & u17_np.isfinite(vd) & (vd > 0)
    t = u17_np.divide(ed, u17_np.sqrt(u17_np.maximum(vd, 0)), out=u17_np.zeros_like(ed), where=valid)
    qc = u17_pd.read_csv(u17_qc_path(sid, task), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    dof = int((qc.n_timepoints - qc.design_rank).sum())
    return (t, valid, e, dof)


In [ ]:
def u17_roi_masks(definition='ALE_3D'):
    if definition != 'ALE_3D':
        raise ValueError('The specified analysis uses independent three-dimensional ALE cluster masks only')
    cfg = u17_json.loads(u17_ROI_CONFIG.read_text())
    manifest = u17_NEW / 'definition_and_provenance.json'
    if not manifest.exists():
        raise FileNotFoundError('Provide and validate independent ALE ROI inputs, or run the ROI definition step in this notebook')
    meta = u17_json.loads(manifest.read_text())
    if meta['config_sha256'] != u17_sha(u17_ROI_CONFIG):
        raise ValueError('ROI definition changed; regenerate ROI masks')
    ref = u17_nib.load(u17_first_path(u17_subjects()[0], 'stroop', 'effect'))
    masks = {}
    for roi in u17_ROIS:
        path = u17_NEW / f'{roi}_ALE_3D.nii.gz'
        if u17_sha(path) != meta['masks'][roi]['sha256']:
            raise ValueError('ALE mask validation failed')
        img = u17_nib.load(path)
        data = u17_np.asarray(img.dataobj)
        if img.shape != ref.shape or not u17_np.allclose(img.affine, ref.affine):
            raise ValueError('ALE ROI and subject statistic image spaces do not match')
        if not u17_np.isin(data, [0, 1]).all() or not data.any():
            raise ValueError('ALE mask must be nonempty and binary')
        masks[roi] = data > 0
    return (masks, ref)


In [ ]:
u12_imaging_paths = u15_imaging_paths
u12_matplotlib.use('Agg')
u12_Path = u17_Path
u12_json = u17_json
u12_os = u17_os
u12_np = u17_np
u12_pd = u17_pd
u12_MRI = u17_MRI
u12_Path = u17_Path
u12_json = u17_json
u12_os = u17_os
u12_np = u17_np
u12_pd = u17_pd
u12_MRI = u17_MRI
u12_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u12_MRI = u17_MRI
u12_matplotlib.use('Agg')
u12_MRI = u17_MRI
u12_Path = u17_Path
u12_json = u17_json
u12_os = u17_os
u12_np = u17_np
u12_pd = u17_pd
u12_MRI = u17_MRI
u13_ROOT = u15_ROOT
u13_DATA = u15_DATA
u13_RESULTS = u15_RESULTS
u13__A8Path = u17__A8Path
u13_Path = u17_Path
u13_sys = u17_sys
u13_hashlib = u17_hashlib
u13_json = u17_json
u13_os = u17_os
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_nilearn = u17_nilearn
u13_stats = u17_stats
u13_multipletests = u17_multipletests
u13_PROJECT = u17_PROJECT
u13_RESULTS = u17_RESULTS
u13_DATA = u17_DATA
u13_EXPECTED_N = u17_EXPECTED_N
u13_participants = u17_participants
u13_source_signature = u17_source_signature
u13_MRI = u17_MRI
u13_TASK = u17_TASK
u13_REST = u17_REST
u13_VERSION = u17_VERSION
u13_LEGACY = u17_LEGACY
u13_NEW = u17_NEW
u13_REST_NEW = u17_REST_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROI_VARIANTS = u17_ROI_VARIANTS
u13_ROIS = u17_ROIS
u13_CONTRAST = u17_CONTRAST
u13_N_PERM = u17_N_PERM
u13_SEED = u17_SEED
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_fingerprint = u17_fingerprint
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_fdr = u17_fdr
u13_mean_stats = u17_mean_stats
u13_first_path = u17_first_path
u13_new_first_complete = u17_new_first_complete
u13_qc_path = u17_qc_path
u13_t_map = u17_t_map
u13_roi_masks = u17_roi_masks
u13__A8Path = u17__A8Path
u13_Path = u17_Path
u13_sys = u17_sys
u13_hashlib = u17_hashlib
u13_json = u17_json
u13_os = u17_os
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_nilearn = u17_nilearn
u13_stats = u17_stats
u13_multipletests = u17_multipletests
u13_PROJECT = u17_PROJECT
u13_RESULTS = u17_RESULTS
u13_DATA = u17_DATA
u13_EXPECTED_N = u17_EXPECTED_N
u13_participants = u17_participants
u13_source_signature = u17_source_signature
u13_MRI = u17_MRI
u13_TASK = u17_TASK
u13_REST = u17_REST
u13_VERSION = u17_VERSION
u13_LEGACY = u17_LEGACY
u13_NEW = u17_NEW
u13_REST_NEW = u17_REST_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROI_VARIANTS = u17_ROI_VARIANTS
u13_ROIS = u17_ROIS
u13_CONTRAST = u17_CONTRAST
u13_N_PERM = u17_N_PERM
u13_SEED = u17_SEED
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_fingerprint = u17_fingerprint
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_fdr = u17_fdr
u13_mean_stats = u17_mean_stats
u13_first_path = u17_first_path
u13_new_first_complete = u17_new_first_complete
u13_qc_path = u17_qc_path
u13_t_map = u17_t_map
u13_roi_masks = u17_roi_masks
u13__A8Path = u17__A8Path
u13_Path = u17_Path
u13_sys = u17_sys
u13_hashlib = u17_hashlib
u13_json = u17_json
u13_os = u17_os
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_nilearn = u17_nilearn
u13_stats = u17_stats
u13_multipletests = u17_multipletests
u13_PROJECT = u17_PROJECT
u13_RESULTS = u17_RESULTS
u13_DATA = u17_DATA
u13_EXPECTED_N = u17_EXPECTED_N
u13_participants = u17_participants
u13_source_signature = u17_source_signature
u13_MRI = u17_MRI
u13_TASK = u17_TASK
u13_REST = u17_REST
u13_VERSION = u17_VERSION
u13_LEGACY = u17_LEGACY
u13_NEW = u17_NEW
u13_REST_NEW = u17_REST_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROI_VARIANTS = u17_ROI_VARIANTS
u13_ROIS = u17_ROIS
u13_CONTRAST = u17_CONTRAST
u13_N_PERM = u17_N_PERM
u13_SEED = u17_SEED
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_fingerprint = u17_fingerprint
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_fdr = u17_fdr
u13_mean_stats = u17_mean_stats
u13_first_path = u17_first_path
u13_new_first_complete = u17_new_first_complete
u13_qc_path = u17_qc_path
u13_t_map = u17_t_map
u13_roi_masks = u17_roi_masks
u13_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u13_ROIS = u17_ROIS
u13_CONTRAST = u17_CONTRAST
u13_MRI = u17_MRI
u13__A8Path = u17__A8Path
u13_Path = u17_Path
u13_sys = u17_sys
u13_hashlib = u17_hashlib
u13_json = u17_json
u13_os = u17_os
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_nilearn = u17_nilearn
u13_stats = u17_stats
u13_multipletests = u17_multipletests
u13_PROJECT = u17_PROJECT
u13_RESULTS = u17_RESULTS
u13_DATA = u17_DATA
u13_EXPECTED_N = u17_EXPECTED_N
u13_participants = u17_participants
u13_source_signature = u17_source_signature
u13_MRI = u17_MRI
u13_TASK = u17_TASK
u13_REST = u17_REST
u13_VERSION = u17_VERSION
u13_LEGACY = u17_LEGACY
u13_NEW = u17_NEW
u13_REST_NEW = u17_REST_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROI_VARIANTS = u17_ROI_VARIANTS
u13_ROIS = u17_ROIS
u13_CONTRAST = u17_CONTRAST
u13_N_PERM = u17_N_PERM
u13_SEED = u17_SEED
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_fingerprint = u17_fingerprint
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_fdr = u17_fdr
u13_mean_stats = u17_mean_stats
u13_first_path = u17_first_path
u13_new_first_complete = u17_new_first_complete
u13_qc_path = u17_qc_path
u13_t_map = u17_t_map
u13_roi_masks = u17_roi_masks
u13_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
u13_sys.dont_write_bytecode = True
u13_TASK = u17_TASK
u13_REST = u17_REST
u13_ROIS = u17_ROIS
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_SUPP_REGIONS = ['whole_brain', *u13_ROIS[:3]]
u13_QC = u13_MRI / 'figure_qa'
for u12_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u12_os.environ.setdefault(u12_key, '1')
for u12_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u12_os.environ.setdefault(u12_key, '1')
u12_group_SYSTEM_FONT = u12_Path('/System/Library/Fonts/Supplemental/Arial.ttf')
if u12_group_SYSTEM_FONT.exists():
    u12_font_manager.fontManager.addfont(u12_group_SYSTEM_FONT)
    u12_plt.rcParams['font.family'] = 'Arial'
u12_brain_CACHE = u12_MRI / 'display_cache'
u12_brain_CACHE.mkdir(exist_ok=True, parents=True)
u13_sys.dont_write_bytecode = True
u13_sys.dont_write_bytecode = True
u13_style_ROOT = u13_MRI
u13_sys.dont_write_bytecode = True


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u13_display = a8_display_three_line


In [ ]:
u13_alignment_overlap()
u13_roi_define()
u13_alignment_dice()
u13_display(u13_pd.read_excel(u13_TASK / 'tables/individual_Dice_whole_brain_and_shared_ROIs_summary.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))
u13_dice_distribution()
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_S2_p001_individual_Dice_distribution_n106.png'), width=650))
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_S2_p001_activation_counts_n106.png'), width=650))
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_S2_p01_individual_Dice_distribution_n106.png'), width=650))
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_S2_p01_activation_counts_n106.png'), width=650))


In [ ]:
u13_roi_activation()
u13_display(u13_pd.read_excel(u13_TASK / 'tables/independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))
u13_roi_patterns()
u13_display(u13_pd.read_excel(u13_TASK / 'tables/independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))
u13_plot_task()
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_independent_ALE_ROI_activation_n106.png'), width=600))
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_independent_ALE_ROI_cross_task_paired_difference_n106.png'), width=600))
u13_display(u13_Image(filename=str(u13_TASK / 'image/Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106.png'), width=600))
u13_pattern_similarity_distribution()
u13_display(u13_Image(filename=str(u13_TASK / 'image' / 'Figure_S7_shared_ROI_cross_task_pattern_similarity_individual_distribution_n106.png')))
